# Классификация проектов с помощью линейной регрессии

In [1]:
# install libraries

%pip install numpy==1.23.5
%pip install typer==0.9.4
%pip install torch==2.0.1
%pip install transformers==4.34.0
%pip install sentence-transformers==3.0.0
%pip install spacy==3.5.4
%pip install tensorflow==2.12.0
%pip install torchtext==0.15.2
%pip install nltk==3.7
%pip install scipy==1.15.3
%pip install gensim==4.4.0
%pip install xgboost==1.7.6
%pip install catboost
%pip install pymorphy3

%pip check

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --up

In [2]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import re
import string

import nltk
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
from gensim.models.word2vec import Word2Vec
from xgboost import XGBClassifier
from catboost import CatBoostClassifier, Pool
from pymorphy3 import MorphAnalyzer

2026-05-11 18:19:54.419690: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


## Функции загрузки и предобработки данных

In [3]:
Labels = [
    "Автомобильные дороги",
    "Водоотведение",
    "Водопроводы",
    "Газоны дорожки",
    "Газопроводы",
    "Горные выработки",
    "Железнодорожные пути",
    "Заводы фабрики",
    "Здания",
    "Инженерное обеспечение",
    "Инфраструктура наземного электротранспорта",
    "Линии электропередачи",
    "Метрополитены",
    "Мосты и тоннели",
    "Наружное освещение",
    "Нефтепроводы",
    "Сооружения",
    "Теплопроводы",
    "Технологические установки",
]

ColumnNames = ["id", "project_name", "label"]

nltk.download('punkt')

def load_labeled_data_raw(path):
    labeled_dataframes = [
        pd.read_csv(f"{path}//{label}.csv", names=ColumnNames, header=0)
        for label in tqdm(Labels)
    ]
    result_df = pd.concat(labeled_dataframes)
    result_df["project_name"] = result_df["project_name"].str.strip('"')
    return result_df

def remove_not_required(df):
    not_required_mask = df['project_name'] != 'Не требуется'
    df = df.loc[not_required_mask]

def split_by_string(text, string):
    split_parts = re.split(string, text, flags=re.IGNORECASE)
    if (len(split_parts) > 1):
        return split_parts[0].strip()
    else:
        return text

# Удаление адресов необходимо, поскольку они добавляют семантического шума.
# Вектора проектов в одном городе будут ближе друг к другу, чем вектора проектов в разных городах.
def remove_adress_substring (df):
    # Порядок важен: "Почтовый адрес" обязателен и стоит в конце, 
    # но помимо этого проект может завершаться на "по адресу" или "адрес объекта" в любом регистре
    df['project_name'] = df['project_name'].apply(lambda text : split_by_string(text, 'Почтовый адрес:'))
    df['project_name'] = df['project_name'].apply(lambda text : split_by_string(text, 'по адресу'))
    df['project_name'] = df['project_name'].apply(lambda text : split_by_string(text, 'адрес'))

def load_labeled_data(path):
    df = load_labeled_data_raw(path)
    remove_not_required(df)
    remove_adress_substring(df)
    return df

def load_unlabeled_data_raw(path):
    return pd.read_csv(path, sep=";", encoding="utf-8", nrows=200000, names=["id", "project_name"], header=0)

def load_unlabeled_data(path):
    df = load_unlabeled_data_raw(path)
    remove_not_required(df)
    remove_adress_substring(df)
    return df

pymorphy_analyzer = MorphAnalyzer()
lemmas_cache = {}

def tokenize_lemmatize(text):
    words = nltk.word_tokenize(text.lower(), language="russian")
    punctuation = set(string.punctuation)
    punctuation.add('``')
    punctuation.add('\'\'')
    result = []
    
    for word in words:
        if word in punctuation:
            pass
        if word in lemmas_cache:
            result.append(lemmas_cache[word])
        else:
            lemma = pymorphy_analyzer.parse(word)[0].normal_form
            lemmas_cache[word] = lemma
            result.append(lemma)
    return result

def tokenize_lemmatize_series(series):
    return series.apply(tokenize_lemmatize)

[nltk_data] Downloading package punkt to /home/jupyter/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## Функции разделения данных на выборки

In [4]:
def data_train_test_split(data, labels):
    assert len(data) == len(
        labels
    ), "Размеры списков данных и результатов разметки не совпадают"
    le = LabelEncoder()
    le.fit(labels)
    y = le.transform(labels)
    return train_test_split(data, y, test_size=0.2, random_state=42)

## Функции векторизации

### Векторизация с помощью Word2Vec

In [5]:
def vectorize_with_word2vec(texts, model, vector_size):
    result = []
    for text in texts:
        text_vector = []
        for word in tokenize_lemmatize(text):
            if word in model.wv:
                text_vector.append(model.wv[word])

        if len(text_vector):
            text_vector = np.average(text_vector, axis=0)
        else:
            text_vector = np.zeros(vector_size)
        result.append(text_vector)
    return result

### Векторизация с помощью Sentence transformer

In [6]:
# Вернет матрицу размера (len(sentences, 1024)
def vectorize_with_sentence_transformer(model_name, sentences):
    model = SentenceTransformer(model_name)
    return model.encode(sentences.to_numpy())

## Функции оптимизации с помощью Grid search и Random search

In [7]:
def find_best_model_gs(X_train, y_train, estimator, param_grid, n_jobs = -1):
    grid_search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring="accuracy",
        cv=3,
        # Этим управляем для CatBoost - при использовании -1 (занять все ядра) он формирует столько параллельных процессов на CPU, 
        # что для них не хватает памяти на GPU
        n_jobs=n_jobs
    )
    grid_search.fit(X_train, y_train)
    return grid_search.best_estimator_, grid_search.best_params_

def find_best_model_rs(X_train, y_train, estimator, param_dist, n_iter, n_jobs = -1):
    random_search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring="accuracy",
        cv=3,
        # Этим управляем для CatBoost - при использовании -1 (занять все ядра) он формирует столько параллельных процессов на CPU, 
        # что для них не хватает памяти на GPU
        n_jobs=n_jobs
    )
    random_search.fit(X_train, y_train)
    return random_search.best_estimator_, random_search.best_params_

## Классификация проектов

### Загрузим данные

In [8]:
df = load_labeled_data("../Data/Reestr/Размеченные")
#df

100%|██████████| 19/19 [00:00<00:00, 117.88it/s]


### Векторизация

In [9]:
# Векторизация Word2Vec
# Для векторизации будем использовать весь корпус слов
df_unlabeled = load_unlabeled_data(f'../Data/Reestr/Реестр 2022-2024 clean.csv')
#df_unlabeled_lemmatized = tokenize_lemmatize_series(df_unlabeled['project_name'][30000:40000])
df_unlabeled_lemmatized = tokenize_lemmatize_series(df_unlabeled['project_name'])
word2vec_model = Word2Vec(df_unlabeled_lemmatized, workers=8, vector_size=300, min_count=3, window=5, epochs=15,)
word2vec_vectors = vectorize_with_word2vec(df['project_name'], word2vec_model, 300)
#word2vec_vectors
#np.shape(word2vec_vectors)

In [10]:
# Векторизация с помощью SentenceTransformer с использованием модели 'sberbank-ai/sbert_large_nlu_ru'
sbert_vectors = vectorize_with_sentence_transformer(
    "sberbank-ai/sbert_large_nlu_ru", df["project_name"]
)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:
np.shape(sbert_vectors)

### Разделeние на тестовую и обучающую выборки

In [13]:
X_train, X_test, y_train, y_test = data_train_test_split(word2vec_vectors, df["label"])

### Логистическая регрессия

#### Параметры модели

In [14]:
log_reg_param_grid = {
        "C": [0.05, 0.1, 1, 10, 50],
        "penalty": ["l1", "l2", 'elasticnet'],
        "solver": ["liblinear", "saga"],
        "max_iter": [500, 1000]}

log_reg_test_param_grid = {
        "C": [1],
        "penalty": ["l1"],
        "solver": ["liblinear"],
        "max_iter": [100]}

 #### Вариант с векторизацией word2vec

In [69]:
# Выбор лучшей модели
lr_initial_model = LogisticRegression(random_state=42)
#lr_best_model, lr_best_params = find_best_model_gs(X_train, y_train, lr_initial_model, log_reg_test_param_grid)
lr_best_model, lr_best_params = find_best_model_rs(X_train, y_train, lr_initial_model, log_reg_param_grid, 30)
#lr_best_model.fit(X_train, y_train)
y_pred = lr_best_model.predict(X_test)

print("Лучшие параметры: ", lr_best_params)
print("Точность:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which 

Лучшие параметры:  {'solver': 'saga', 'penalty': 'l2', 'max_iter': 500, 'C': 10}
Точность: 0.8789473684210526
              precision    recall  f1-score   support

           0       0.71      0.89      0.79        19
           1       0.74      0.78      0.76        18
           2       0.88      0.68      0.77        22
           3       1.00      0.83      0.91        24
           4       1.00      0.87      0.93        23
           5       0.92      0.92      0.92        25
           6       1.00      0.88      0.94        17
           7       0.88      0.94      0.91        16
           8       0.52      1.00      0.69        11
           9       0.88      0.96      0.92        23
          10       0.83      0.91      0.87        11
          11       1.00      0.80      0.89        20
          12       1.00      0.95      0.97        20
          13       0.95      0.86      0.90        22
          14       0.96      1.00      0.98        25
          15       0.90  

/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


 #### Вариант с векторизацией sber sentence transformer

In [ ]:
# Выбор лучшей модели
lr_initial_model = LogisticRegression(random_state=42)
#lr_best_model, lr_best_params = find_best_model_gs(X_train, y_train, lr_initial_model, log_reg_test_param_grid)
lr_best_model, lr_best_params = find_best_model_rs(X_train, y_train, lr_initial_model, log_reg_param_grid, 20)
#lr_best_model.fit(X_train, y_train)
y_pred = lr_best_model.predict(X_test)

print("Лучшие параметры: ", lr_best_params)
print("Точность:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

### XGBoost

#### Параметры модели

In [15]:
xgb_test_param_grid = {"n_estimators": [300], "max_depth": [6], "learning_rate": [0.1]}

xgb_param_grid = {
    "n_estimators": [100, 150, 200, 300, 400],
    "max_depth": [3, 4, 6, 8],
    "learning_rate": [0.05, 0.1, 0.2, 0.3],
    # доля случайных взятых в обучение сэмплов
    "subsample" : [0.5, 0.6, 0.7, 0.8, 0.9],
    # регуляризации
    "reg_alpha": [0, 0.01, 0.1, 1, 5],
    "reg_lambda": [0, 0.01, 0.1, 1, 5]
}

#### Вариант с векторизацией sber sentence transformer

In [16]:
xgb_initial_model = XGBClassifier(n_jobs=-1, eval_metric="logloss", tree_method="gpu_hist", random_state=42)
#xgb_best_model, xgb_best_params = find_best_model_gs(X_train, y_train, xgb_initial_model, xgb_param_grid)
xgb_best_model, xgb_best_params = find_best_model_rs(X_train, y_train, xgb_initial_model, xgb_param_grid, 40)
xgb_best_model.fit(X_train, y_train)
y_pred = xgb_best_model.predict(X_test)

#print("Лучшие параметры: ", xgb_best_params)
print("Точность:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Точность: 0.8605263157894737
              precision    recall  f1-score   support

           0       0.61      0.89      0.72        19
           1       0.68      0.83      0.75        18
           2       0.79      0.50      0.61        22
           3       0.95      0.79      0.86        24
           4       1.00      0.87      0.93        23
           5       0.86      0.96      0.91        25
           6       1.00      0.94      0.97        17
           7       0.88      0.94      0.91        16
           8       0.56      0.91      0.69        11
           9       0.92      0.96      0.94        23
          10       0.83      0.91      0.87        11
          11       1.00      0.85      0.92        20
          12       0.90      0.90      0.90        20
          13       1.00      0.82      0.90        22
          14       1.00      1.00      1.00        25
          15       0.95      1.00      0.97        18
          16       0.83      0.58      0.68        2

In [17]:
print("Лучшие параметры: ", xgb_best_params)

Лучшие параметры:  {'subsample': 0.6, 'reg_lambda': 5, 'reg_alpha': 0.01, 'n_estimators': 400, 'max_depth': 4, 'learning_rate': 0.2}


In [ ]:
print("Точность:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

### CatBoost

#### Параметры модели

In [18]:
cb_param_grid = {
        "learning_rate": [0.01, 0.05, 0.1, 0.5],
        "depth": [3, 4, 6, 8],
        "l2_leaf_reg": [1, 2, 4, 8, 10],
        "iterations": [50, 100, 300, 500],
        "random_strength": [0.5, 1, 2.0],
        "bagging_temperature": [0.8, 1.0, 1.2]}

cb_test_param_grid = {"learning_rate": [0.1]}

#### Вариант с векторизацией sber sentence transformer

In [ ]:
!nvidia-smi --query-gpu=memory.total,memory.used,memory.free --format=csv

In [19]:
cb_initial_model = CatBoostClassifier(logging_level='Silent', devices='0', gpu_ram_part=0.3, loss_function='MultiClass', task_type="GPU", random_seed=42)
#cb_best_model, cb_best_params = find_best_model_rs(X_train, y_train, cb_initial_model, cb_test_param_grid)
cb_best_model, cb_best_params = find_best_model_rs(X_train, y_train, cb_initial_model, cb_param_grid, 40, n_jobs = 3)

#cb_best_model.fit(X_train, y_train)
y_pred = cb_best_model.predict(X_test)

print("Лучшие параметры: ", cb_best_params)
print("Точность:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Лучшие параметры:  {'random_strength': 0.5, 'learning_rate': 0.1, 'l2_leaf_reg': 2, 'iterations': 500, 'depth': 6, 'bagging_temperature': 1.2}
Точность: 0.8421052631578947
              precision    recall  f1-score   support

           0       0.71      0.79      0.75        19
           1       0.63      0.67      0.65        18
           2       0.59      0.45      0.51        22
           3       0.95      0.75      0.84        24
           4       0.91      0.87      0.89        23
           5       0.96      0.92      0.94        25
           6       1.00      0.94      0.97        17
           7       0.78      0.88      0.82        16
           8       0.56      0.91      0.69        11
           9       0.83      0.87      0.85        23
          10       0.79      1.00      0.88        11
          11       1.00      0.85      0.92        20
          12       0.90      0.95      0.93        20
          13       0.90      0.86      0.88        22
          14     